# 2026.10.15(목) 19h~

## Seq2Seq LSTM + Teacher Forcing + Embedding + Attention

글자 알파벳 문자열을 입력받아 거꾸로 출력하는 예제입니다.

적용 개념: Seq2Seq, Teacher Forcing, Embedding, Attention\n

In [1]:
# ============================================================
# Seq2Seq LSTM + Teacher Forcing + Embedding + Attention 적용 버전
# ------------------------------------------------------------
# 목적:
#   알파벳 대소문자로 구성된 문자열을 입력받아
#   그 문자열을 거꾸로 뒤집어 출력하는 모델을 학습한다.
#
# 예)
#   입력: AbCdE
#   출력: EdCbA
#
# 현재 설정:
#   cal_len = 52
#   즉, 입력 문자열은 반드시 52글자여야 한다.
#
# 이 코드에 포함된 핵심 개념:
#   1. Seq2Seq
#   2. Teacher Forcing
#   3. Embedding
#   4. LSTM Encoder-Decoder
#   5. Attention
# ============================================================


import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Attention, Concatenate
from tensorflow.keras.callbacks import EarlyStopping


# ------------------------------------------------------------
# 1. 기본 설정
# ------------------------------------------------------------

# 하나의 입력 문자열 길이
# cal_len = 52이면 한 샘플은 52글자 문자열이다.
# 예: ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
cal_len = 52

# 학습 데이터 샘플 수
# 랜덤 문자열 20,000개를 생성해서 학습에 사용한다.
num_samples = 20000

# 난수 고정
# 실행할 때마다 같은 랜덤 데이터가 생성되도록 설정한다.
# 완전히 동일한 결과를 보장하지는 않지만 재현성 확보에 도움된다.
np.random.seed(0)
tf.random.set_seed(0)


# 사용할 문자 집합
# 대문자 26개 + 소문자 26개 = 총 52개 문자
chars = 'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz'

# 실제 문자 개수
num_chars = len(chars)          # 52


# START_TOKEN
# ------------------------------------------------------------
# Teacher Forcing에서 디코더 입력의 첫 번째 위치에 넣는 시작 토큰이다.
#
# 실제 문자는 0~51 인덱스를 사용한다.
# 따라서 START_TOKEN은 실제 문자와 겹치지 않도록 52번을 사용한다.
#
# 예)
#   A → 0
#   B → 1
#   ...
#   z → 51
#   START → 52
# ------------------------------------------------------------
START_TOKEN = num_chars         # 52


# vocab 크기 설정
# ------------------------------------------------------------
# encoder_vocab_size:
#   인코더는 실제 입력 문자만 받는다.
#   따라서 A~Z, a~z 총 52개만 필요하다.
#
# decoder_vocab_size:
#   디코더 입력에는 실제 문자 + START_TOKEN이 들어간다.
#   따라서 52개 문자 + START 1개 = 53개가 필요하다.
#
# output_classes:
#   모델이 최종적으로 예측해야 하는 것은 실제 문자이다.
#   START_TOKEN은 출력 정답에 나오면 안 된다.
#   따라서 출력 클래스 수는 52개이다.
# ------------------------------------------------------------
encoder_vocab_size = num_chars      # 52
decoder_vocab_size = num_chars + 1  # 53
output_classes = num_chars          # 52


# 문자 → 숫자 인덱스 변환 딕셔너리
char_to_index = {char: idx for idx, char in enumerate(chars)}

# 숫자 인덱스 → 문자 변환 딕셔너리
index_to_char = {idx: char for idx, char in enumerate(chars)}


# ------------------------------------------------------------
# 2. 데이터 생성 함수
# ------------------------------------------------------------

def generate_data(num_samples):
    """
    랜덤 문자열 데이터를 생성하는 함수

    X:
        원래 문자열의 문자 인덱스 배열

    y:
        X를 거꾸로 뒤집은 정답 문자열의 문자 인덱스 배열

    예)
        원래 문자열: AbCdE
        X = [0, 27, 2, 29, 4]

        뒤집은 문자열: EdCbA
        y = [4, 29, 2, 27, 0]
    """

    # 빈 리스트 생성
    X = []
    y = []

    for _ in range(num_samples): # 활용 안 하는 건 _로 표기

        # chars에서 cal_len개 문자를 랜덤 선택한다.
        # cal_len = 52이면 52글자 문자열 생성
        sequence = np.random.choice(list(chars), cal_len)

        # 입력 문자열을 문자 인덱스로 변환
        X.append([char_to_index[char] for char in sequence])

        # 정답은 입력 문자열을 뒤집은 값
        # sequence[::-1]은 파이썬에서 순서를 거꾸로 뒤집는 문법이다.
        y.append([char_to_index[char] for char in sequence[::-1]])

    return np.array(X), np.array(y) # numpy array로 변환


# ------------------------------------------------------------
# 3. 데이터 생성
# ------------------------------------------------------------

# X: 인코더에 들어갈 입력 데이터
# y: 디코더가 맞혀야 할 정답 데이터
X, y = generate_data(num_samples)

# 현재 shape 예시
# X.shape = (20000, 52)
# y.shape = (20000, 52)


# ------------------------------------------------------------
# Teacher Forcing용 디코더 입력 생성
# ------------------------------------------------------------
# Seq2Seq에서 디코더는 이전에 나온 값을 보고 다음 값을 예측한다.
#
# 정답 y가 다음과 같다고 하면:
#   y = [E, d, C, b, A]
#
# 디코더 입력은 다음처럼 만든다.
#   decoder_input_data = [START, E, d, C, b]
#
# 즉, 정답 y를 그대로 넣는 것이 아니라
# 한 칸 오른쪽으로 밀어서 넣는다.
#
# 학습 관계:
#   START → E 예측
#   E     → d 예측
#   d     → C 예측
#   C     → b 예측
#   b     → A 예측
# ------------------------------------------------------------

# y와 같은 shape의 배열을 0으로 생성
decoder_input_data = np.zeros_like(y) # y처럼 생긴..

# 모든 샘플의 첫 번째 위치에 START_TOKEN 입력
decoder_input_data[:, 0] = START_TOKEN

# 두 번째 위치부터는 y의 마지막 전까지 값을 복사
# y[:, :-1]은 y에서 마지막 글자를 제외한 값
# decoder_input_data[:, 1:]은 첫 번째 칸을 제외한 나머지 칸
decoder_input_data[:, 1:] = y[:, :-1]


# ------------------------------------------------------------
# 4. Seq2Seq + Attention 모델 정의
# ------------------------------------------------------------

# 문자 하나를 몇 차원 벡터로 표현할지 결정
# Embedding Layer가 각 문자 인덱스를 32차원 벡터로 변환한다 -> 강사님께서 19h25쯤? 왜 32가 아닌 크기는(52?) 비효율적인지 설명해 주셨는데, 정확히 못 들음..
embedding_dim = 32

# LSTM hidden state 크기
# 인코더와 디코더 LSTM의 내부 상태 벡터 크기
hidden_units = 128


# -----------------------------
# 4-1. 인코더
# -----------------------------

# 인코더 입력층
# Embedding을 사용하므로 입력 shape는 (cal_len,)이다.
# 즉, 한 샘플은 52개의 정수 인덱스로 구성된다.
encoder_inputs = Input(shape=(cal_len,), name="encoder_inputs")


# 인코더 Embedding Layer
# ------------------------------------------------------------
# 기존처럼 문자 인덱스를 그대로 LSTM에 넣으면
# A=0, B=1, z=51처럼 숫자 크기 관계가 있는 것처럼 처리될 수 있다.
#
# Embedding을 사용하면 각 문자를 학습 가능한 벡터로 바꾼다.
#
# 입력 shape:
#   (batch_size, cal_len)
#
# 출력 shape:
#   (batch_size, cal_len, embedding_dim)
# ------------------------------------------------------------
encoder_embedding = Embedding(
    input_dim=encoder_vocab_size,
    output_dim=embedding_dim,
    name="encoder_embedding"
)(encoder_inputs)


# 인코더 LSTM
# ------------------------------------------------------------
# return_sequences=True:
#   Attention을 적용하려면 인코더의 마지막 출력(hidden state)만 있으면 안 된다.
#   디코더가 인코더의 모든 시점 출력을 참고해야 하므로
#   전체 시점의 hidden state를 반환해야 한다.
#
# return_state=True:
#   마지막 hidden state(state_h)와 cell state(state_c)를 반환한다.
#   이 두 상태는 디코더 LSTM의 초기 상태/입력으로 전달된다 -> 나의 질문 = 하나만 전달하면 what happens?
#
# [답변 - "state를 하나만(h만 또는 c만) 전달하면?"]
#  - LSTM의 "상태"는 (h, c) 두 개가 한 쌍이다. h = 단기 기억(= 그 시점의 출력),
#    c = LSTM 안쪽에만 있는 장기 기억(출력으로는 드러나지 않음).
#    그래서 디코더의 initial_state는 반드시 [h, c] 두 개짜리 리스트여야 한다.
#  - 실제로 하나만 넘겨 보면(voice-ai 환경에서 확인): 모델을 "만드는" 단계에서는 통과하지만
#    predict/fit을 "실행"할 때 `list index out of range`라는 알아보기 어려운 오류가 난다.
#    (LSTM이 두 번째 상태를 꺼내려는데 없기 때문)
#  - 오류를 피하려고 한쪽을 0으로 채워서 [h, 0]을 주면 돌아는 가지만, 인코더가 쌓은 c(장기 기억)
#    정보가 디코더에 전달되지 않는다. 성능이 얼마나 떨어지는지는 실험해 봐야 안다.
#  - 아예 안 주면(initial_state=None) 디코더는 h=0, c=0의 "빈 기억"에서 출발한다. 오류는 없다.
#    이 모델은 Attention이 인코더 전체 출력을 따로 참조하므로 그래도 학습은 되지만,
#    인코더가 읽은 결과를 디코더 출발점으로 직접 넘겨주는 통로가 끊긴다.
#
# [부연 - "Attention이 인코더 전체 출력을 따로 참조한다"는 말의 뜻]
#  인코더에서 디코더로 정보가 가는 통로가 이 모델에는 "두 개" 있다.
#
#   통로 1) 마지막 상태 (state_h, state_c) → 디코더의 initial_state
#           · 입력 52글자를 다 읽은 "마지막 시점"의 요약 한 덩어리(h, c 각 128차원)만 넘어간다.
#           · 디코더가 출발할 때 한 번만 받는다. (기존 Seq2Seq가 쓰던 유일한 통로 = 병목)
#
#   통로 2) 모든 시점의 출력 encoder_outputs → Attention 층
#           · 인코더는 글자를 한 칸 읽을 때마다 출력을 하나씩 낸다. 그 52개 전부가 encoder_outputs
#             (shape: 52 x 128)이고, return_sequences=True라서 버려지지 않고 남아 있다.
#           · Attention 층 입력 [decoder_outputs, encoder_outputs]의 두 번째 인자가 바로 이것이다.
#           · 디코더는 글자를 하나 낼 때마다 이 52개를 "직접 훑어보고" 필요한 위치에 가중치를 둔다.
#             → 이것이 "따로 참조한다"는 뜻: 마지막 상태 하나를 거치지 않고, 입력의 각 위치 정보를 그대로 꺼내 본다.
#
#  구체적인 예 (5글자로 줄여서): 입력 AbCdE, 정답 EdCbA
#      encoder_outputs = [o1(A까지 읽은 출력), o2(b까지), o3(C까지), o4(d까지), o5(E까지)]
#      · 디코더가 첫 글자 'E'를 낼 때: 점수를 계산해 보니 o5에 가중치 0.9, 나머지에 0.1을 나눠 줌
#        → 컨텍스트 벡터는 거의 o5(E를 읽은 시점의 정보). 입력의 맨 끝 글자를 직접 가져온 셈이다.
#      · 두 번째 글자 'd'를 낼 때: 가중치가 o4로 옮겨 감 … (역순 과제라면 이런 식으로 반대 방향 대각선 패턴이
#        학습될 것으로 예상된다. 실제 가중치는 확인해 보지 않았다.)
#  그래서 initial_state를 안 주어도(통로 1을 끊어도) 디코더는 통로 2로 입력 내용을 볼 수 있다.
#  반대로 통로 1만 있으면 52글자 정보를 128차원 벡터 하나에 눌러 담아야 한다.
#  (참고: 단방향 LSTM이라 o_t에는 "t번째 글자까지 읽은" 정보가 담긴다. 앞쪽 위치 출력에는 뒤쪽 글자 정보가 없다.)
# ------------------------------------------------------------
encoder_lstm = LSTM(
    hidden_units,
    return_sequences=True,
    return_state=True,
    name="encoder_lstm"
)


# encoder_outputs:
#   인코더의 모든 시점 hidden state
#   shape: (batch_size, cal_len, hidden_units)
#
# state_h:
#   인코더 마지막 시점의 hidden state
#
# state_c:
#   인코더 마지막 시점의 cell state
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding) # 나의 질문 = encoder_lstm 위에서 정의?한 것과 여기서 호출/사용?하는 코드 형태 각각 정확히 이해 안 됨 + 왜 encoder_lstm에는 초기상태 전달 안 하지..? 하면 what happens?
#
# [답변 1 - "정의"와 "호출"은 각각 무엇인가?]
#  - 위에서 한 것:  encoder_lstm = LSTM(hidden_units, return_sequences=True, return_state=True, ...)
#      → LSTM "층 객체"를 만든다. 공장에 기계를 한 대 설치하는 것과 같다.
#        이 시점에는 유닛 수(128), 옵션(return_*) 같은 설정만 들어 있고 데이터와는 연결돼 있지 않다.
#  - 여기서 하는 것:  encoder_lstm(encoder_embedding)
#      → 그 층 객체를 "함수처럼 호출"해서 입력(encoder_embedding)을 연결한다. 기계에 재료를 넣는 것과 같다.
#        Keras 층은 객체이면서 호출 가능(callable)해서 이렇게 쓴다. 이 방식이 Functional API다.
#        (Functional API에서는 진짜 데이터가 아니라 "자리표시 텐서"가 흘러가며 계산 그래프가 이어진다.
#         진짜 데이터는 나중에 model.fit([...])으로 들어온다.)
#  - 반환값이 3개인 이유: return_sequences=True → 모든 시점의 출력(encoder_outputs),
#      return_state=True → 마지막 h(state_h)와 c(state_c).
#  - 둘로 나눠 쓰는 이유: 한 줄로 LSTM(...)(x)처럼 써도 똑같다. 다만 층 객체를 변수에 담아 두면
#      같은 층을 여러 번 호출해 가중치를 공유할 수도 있고, 이 코드처럼 읽기도 쉽다.
#      (디코더 LSTM은 별도 객체 decoder_lstm이라 인코더와 가중치가 따로다.)
#
# [답변 2 - "왜 인코더에는 initial_state를 안 주나? 주면?"]
#  - 인코더는 전체 흐름의 "맨 처음"이라서, 넘겨받을 이전 기억이 없다. initial_state를 안 주면
#    h=0, c=0(빈 기억)에서 시작하는데, 입력을 처음 읽는 인코더에게는 이게 자연스러운 출발점이다.
#  - 인코더에 initial_state를 준다면 줄 값이 필요하다. 0을 주면 기본값과 똑같고, 랜덤값이나 디코더의
#    상태를 주는 건 의미가 없다(디코더 상태는 인코더 결과에서 만들어지는 값이라 순서가 거꾸로다).
#  - 반대로 디코더는 "인코더가 입력을 읽고 만든 기억"을 받아야 하므로 initial_state=encoder_states를 준다.
#    인코더 → 디코더 정보 통로가 바로 이 initial_state다.


# 디코더 초기 상태로 사용할 인코더 상태
encoder_states = [state_h, state_c]


# -----------------------------
# 4-2. 디코더
# -----------------------------

# 디코더 입력층
# Teacher Forcing으로 만든 decoder_input_data가 들어간다.
decoder_inputs = Input(shape=(cal_len,), name="decoder_inputs")


# 디코더 Embedding Layer
# ------------------------------------------------------------
# 디코더 입력에는 실제 문자 52개뿐 아니라 START_TOKEN도 들어간다.
# 그래서 input_dim은 decoder_vocab_size = 53이다.
# ------------------------------------------------------------
decoder_embedding = Embedding(
    input_dim=decoder_vocab_size,
    output_dim=embedding_dim,
    name="decoder_embedding"
)(decoder_inputs)


# 디코더 LSTM
# ------------------------------------------------------------
# return_sequences=True:
#   디코더는 한 글자만 출력하는 것이 아니라
#   cal_len개의 출력 시퀀스를 만들어야 한다.
#
# return_state=True:
#   디코더의 마지막 상태도 반환한다.
#   현재 학습 모델에서는 반환된 상태를 직접 사용하지 않지만,
#   Seq2Seq 구조상 추론 모델을 분리할 때 자주 사용된다.
# ------------------------------------------------------------
decoder_lstm = LSTM(
    hidden_units,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)


# 디코더 실행
# ------------------------------------------------------------
# decoder_embedding:
#   Teacher Forcing용 디코더 입력
#
# initial_state=encoder_states:
#   인코더가 입력 문자열을 읽고 만든 마지막 상태를
#   디코더의 초기 상태로 사용한다.
#
# decoder_outputs:
#   디코더의 모든 시점 hidden state
#   shape: (batch_size, cal_len, hidden_units)
# ------------------------------------------------------------
decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)


# -----------------------------
# 4-3. Attention
# -----------------------------

# Attention의 역할
# ------------------------------------------------------------
# 기본 Seq2Seq는 인코더의 마지막 상태만 디코더에 넘긴다.
# 하지만 입력이 길어지면 마지막 상태 하나에 모든 정보를 담기 어렵다.
#
# Attention은 디코더가 각 글자를 출력할 때마다
# 인코더의 전체 시점 출력 중 어느 위치를 더 참고할지 계산한다
# attention의 역할: 나의 답변 = decorder와 더 연관있는 인코더의 정보로 context vector를 만들게 됨 vs 강사님 설명 = attention을 통해 '역순'이라는 순서를 학습함
#
# 이 코드에서:
#   query = decoder_outputs
#   key   = encoder_outputs
#   value = encoder_outputs
#
# 즉, 디코더의 현재 상태가 query가 되고,
# 인코더의 각 시점 출력이 key/value가 된다.
# ------------------------------------------------------------
attention_layer = Attention(name="attention_layer")


# context_vectors
# ------------------------------------------------------------
# 디코더의 각 시점마다
# 인코더의 모든 시점 출력에 대한 attention score를 계산하고,
# 그 score를 바탕으로 인코더 출력들을 가중합한 context vector를 만든다.
#
# shape:
#   decoder_outputs: (batch_size, decoder_len, hidden_units)
#   encoder_outputs: (batch_size, encoder_len, hidden_units)
#   context_vectors: (batch_size, decoder_len, hidden_units)
# ------------------------------------------------------------
# [답변 - "Attention 입력 순서를 바꾸면 안 되나? 왜 이 순서?"]
#  - Keras Attention 층은 입력을 [query, value] 순서로 받는다. (key를 따로 안 주면 value가 key 역할도 함)
#      · query = "질문하는 쪽" → 지금 디코더가 알고 싶은 것 → decoder_outputs
#      · value(=key) = "찾아볼 자료" → 인코더가 읽은 입력의 모든 위치 → encoder_outputs
#  - 계산: query와 key의 내적으로 점수 (batch, 디코더 길이, 인코더 길이) → softmax로 가중치
#      → value를 가중합 → context (batch, 디코더 길이, hidden_units). 결과 길이는 query의 길이를 따른다.
#  - 순서를 바꾸면: query=encoder_outputs, value=decoder_outputs가 되어 "인코더 위치들이 디코더 출력을
#    훑어보는" 반대 방향이 된다. 의미가 달라져서 "디코더가 입력을 참조한다"는 attention 목적에 어긋난다.
#  - 더 위험한 점: 이 노트북은 인코더·디코더 길이가 둘 다 52라서 순서를 바꿔도 shape 오류 없이 돌아간다
#    (작은 모델로 확인: 어느 순서든 출력 shape이 같음). 길이가 서로 다르면 출력 길이가 인코더 길이로
#    나와서 최종 출력과 안 맞게 되지만, 같은 길이면 조용히 잘못 학습된다. 순서는 반드시 [디코더, 인코더].
context_vectors = attention_layer(
    [decoder_outputs, encoder_outputs] # 나의 질문 = 이 파라미터들 순서 바꾸면 안 되나? 왜 이 순서이지?
)


# 디코더 출력과 Attention Context 결합
# ------------------------------------------------------------
# decoder_outputs:
#   디코더 LSTM이 만든 현재 출력 정보
#
# context_vectors:
#   Attention이 인코더 전체 출력에서 가져온 참고 정보
#
# 두 벡터를 이어 붙여서 최종 예측에 사용한다.
#
# shape:
#   decoder_outputs   : (batch_size, cal_len, 128)
#   context_vectors   : (batch_size, cal_len, 128)
#   결합 후            : (batch_size, cal_len, 256)
# ------------------------------------------------------------
decoder_combined_context = Concatenate(name="concat_decoder_attention")(
    [decoder_outputs, context_vectors]
)


# -----------------------------
# 4-4. 출력층
# -----------------------------

# 각 시점마다 52개 문자 중 하나를 예측한다.
# softmax는 각 문자에 대한 확률을 출력한다.
decoder_dense = Dense(
    output_classes,
    activation='softmax',
    name="decoder_output_dense"
)


# 최종 출력
# shape:
#   (batch_size, cal_len, output_classes)
#   예: (128, 52, 52)
decoder_outputs_final = decoder_dense(decoder_combined_context) # 분류 동작


# ------------------------------------------------------------
# 전체 모델 정의
# ------------------------------------------------------------
# 입력:
#   1. encoder_inputs
#   2. decoder_inputs
#
# 출력:
#   decoder_outputs_final
#
# 학습 시:
#   model.fit([X, decoder_input_data], y)
# ------------------------------------------------------------
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs_final
)


# ------------------------------------------------------------
# 모델 컴파일
# ------------------------------------------------------------
# sparse_categorical_crossentropy:
#   정답 y가 원-핫 인코딩이 아니라 정수 인덱스이므로 사용한다.
#
# 예)
#   정답 A = 0
#   정답 z = 51
#
# metrics=['accuracy']:
#   문자 단위 정확도를 계산한다.
#   문자열 전체가 맞았는지의 정확도는 아래 검증 코드에서 별도로 계산한다.
# ------------------------------------------------------------
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 모델의 요약을 보기 위해 한 번 모델을 호출하여 초기화합니다
# 입력 (샘플 수 미정, 길이 cal_len)로 모델을 미리 만들어 summary()를 볼 수 있게 한다
model.build(input_shape=(None, cal_len))

# 모델 구조 출력
model.summary()


# ------------------------------------------------------------
# 5. 모델 학습
# ------------------------------------------------------------

# EarlyStopping
# ------------------------------------------------------------
# val_loss를 관찰하다가 5번 연속 개선되지 않으면 학습을 멈춘다.
#
# restore_best_weights=True:
#   학습이 끝난 뒤 가장 val_loss가 좋았던 시점의 가중치로 복원한다.
#   따라서 fit() 이후 model.save()를 하면 best weight가 저장된다.
# ------------------------------------------------------------
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

2026-10-15 19:46:01.065259: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 52)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 52)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 52, 32)    │      1,664 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 52, 32)    │      1,696 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 52, 128), │     82,432 │ encoder_embeddin… │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 52, 128), │     82,432 │ decoder_embeddin… │
│                     │ (None, 128),      │            │ encoder_lstm[0][… │
│                     │ (None, 128)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_layer     │ (None, 52, 128)   │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat_decoder_att… │ (None, 52, 256)   │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ attention_layer[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_output_den… │ (None, 52, 52)    │     13,364 │ concat_decoder_a… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 181,588 (709.33 KB)

 Trainable params: 181,588 (709.33 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# 모델 학습
# ------------------------------------------------------------
# 입력:
#   [X, decoder_input_data]
#
# 정답:
#   y
#
# X:
#   인코더 입력
#
# decoder_input_data:
#   Teacher Forcing용 디코더 입력
#
# y:
#   디코더가 최종적으로 맞혀야 하는 정답
# ------------------------------------------------------------
history = model.fit(
    [X, decoder_input_data],
    y,
    epochs=100,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)


# 모델 저장
# ------------------------------------------------------------
# EarlyStopping에서 restore_best_weights=True를 사용했으므로
# 현재 model은 val_loss 기준 best weight로 복원된 상태이다.
# 따라서 이 시점에 저장하면 best model을 저장하는 효과가 있다.
# ------------------------------------------------------------
model.save("seq2seq_teacher_forcing_attention_best.keras")

In [ ]:
# ------------------------------------------------------------
# 6. 예측 함수
# ------------------------------------------------------------

def predict_reverse(input_sequence):
    """
    학습된 모델을 이용해 문자열을 뒤집어 예측하는 함수

    학습 시에는 Teacher Forcing을 사용하므로
    디코더 입력으로 실제 정답의 이전 글자를 넣을 수 있다.

    하지만 예측 시에는 정답을 알 수 없다.

    따라서 예측 시에는:
        1. 첫 번째 디코더 입력에 START_TOKEN을 넣는다.
        2. 첫 번째 글자를 예측한다.
        3. 예측한 글자를 다음 디코더 입력으로 넣는다.
        4. 이 과정을 cal_len번 반복한다.
    """

    # 입력 길이 검사
    # 모델은 cal_len 길이로 학습되었기 때문에
    # 예측할 때도 반드시 같은 길이의 문자열을 입력해야 한다.
    if len(input_sequence) != cal_len:
        raise ValueError(f"입력 문자열은 반드시 {cal_len}글자여야 합니다.")

    # 지원 문자 검사
    # 현재 모델은 A~Z, a~z만 학습했다.
    for char in input_sequence:
        if char not in char_to_index:
            raise ValueError(f"지원하지 않는 문자입니다: {char}")

    # 입력 문자열을 인덱스로 변환
    # 예)
    #   AbCdE → [0, 27, 2, 29, 4]
    input_indices = np.array(
        [char_to_index[char] for char in input_sequence]
    ).reshape((1, cal_len))

    # 디코더 입력 배열 생성
    # 처음에는 전부 0으로 만들고,
    # 첫 번째 위치에 START_TOKEN을 넣는다.
    decoder_input = np.zeros((1, cal_len), dtype=np.int32)
    decoder_input[0, 0] = START_TOKEN

    # 예측된 문자 인덱스를 저장할 리스트
    result_indices = []

    # 한 글자씩 순차적으로 예측
    for t in range(cal_len):

        # 현재까지 채워진 decoder_input을 사용해 전체 시퀀스 예측
        predicted_sequence = model.predict(
            [input_indices, decoder_input],
            verbose=0
        )

        # t번째 위치의 softmax 결과 중 가장 확률이 높은 문자 선택
        pred_idx = np.argmax(predicted_sequence[0, t])

        # 예측 결과 저장
        result_indices.append(pred_idx)

        # 다음 시점의 디코더 입력으로 현재 예측값 사용
        # Teacher Forcing 학습 때는 실제 정답을 넣었지만,
        # 예측 때는 정답을 모르므로 모델이 예측한 값을 넣는다.
        if t + 1 < cal_len:
            decoder_input[0, t + 1] = pred_idx

    # 인덱스를 다시 문자로 변환해서 문자열로 반환
    return ''.join(index_to_char[idx] for idx in result_indices)


# ------------------------------------------------------------
# 7. 검증
# ------------------------------------------------------------

# 테스트용 랜덤 문자열 30개 생성
test_strings = [
    "".join(np.random.choice(list(chars), cal_len))
    for _ in range(30)
]


# 실제 정답 생성
# 각 문자열을 거꾸로 뒤집은 값
expected_outputs = [s[::-1] for s in test_strings]


# 문자열 전체 정확도 계산용 변수
correct_predictions = 0
total_predictions = len(test_strings)


# 테스트 문자열 하나씩 예측
for i, test_string in enumerate(test_strings):

    # 모델 예측 결과
    predicted_output = predict_reverse(test_string)

    # 실제 정답
    expected_output = expected_outputs[i]

    # 문자열 전체가 정확히 일치하는지 확인
    # 한 글자라도 틀리면 False
    is_correct = predicted_output == expected_output

    if is_correct:
        correct_predictions += 1

    print(f"입력: {test_string}")
    print(f"예측된 출력: {predicted_output}")
    print(f"실제 출력: {expected_output}")
    print(f"정확 여부: {'맞음' if is_correct else '틀림'}")
    print("-" * 40)


# 문자열 단위 정확도
# model.fit의 accuracy는 문자 단위 정확도이고,
# 여기서 계산하는 sequence_accuracy는 문자열 전체 정확도이다.
sequence_accuracy = correct_predictions / total_predictions

print(f"문자열 단위 정확도: {sequence_accuracy * 100:.2f}%")
print(f"마지막 val_accuracy는 문자 단위 정확도입니다: {history.history['val_accuracy'][-1]:.4f}")

In [ ]:
# ------------------------------------------------------------
# 8. 직접 입력 테스트
# ------------------------------------------------------------

while True:

    # 사용자로부터 직접 문자열 입력
    sample = input(f"\n테스트할 문자열을 입력하세요. 반드시 {cal_len}글자여야 합니다. 종료하려면 q 입력: ")

    # q 입력 시 종료
    if sample.lower() == "q":
        print("테스트를 종료합니다.")
        break

    # 입력 길이 검사
    if len(sample) != cal_len:
        print(f"입력 길이 오류: 현재 {len(sample)}글자입니다. 반드시 {cal_len}글자를 입력해야 합니다.")
        continue

    # 지원하지 않는 문자가 있는지 검사
    invalid_chars = [char for char in sample if char not in char_to_index]

    if invalid_chars:
        print(f"지원하지 않는 문자가 포함되어 있습니다: {invalid_chars}")
        print("사용 가능한 문자는 A~Z, a~z 입니다.")
        continue

    # 모델 예측
    predicted = predict_reverse(sample)

    # 실제 정답
    expected = sample[::-1]

    print("\n직접 테스트 결과")
    print(f"입력: {sample}")
    print(f"예측: {predicted}")
    print(f"정답: {expected}")
    print(f"정확 여부: {'맞음' if predicted == expected else '틀림'}")

## Teacher Forcing용 디코더 입력, 왜 "정답을 한 칸 오른쪽으로 밀어서" 넣을까?

### 한 문장 요약
> 디코더는 **"앞에서 나온 글자(정답)를 보고, 다음 글자를 맞히는"** 방식으로 학습한다. 그래서 디코더에 넣는 입력은 정답을 **한 칸 뒤로 밀고 맨 앞에 START를 붙인 것**이다.

### 스토리로 이해하기: 받아쓰기 연습
입력 `AbCdE`를 받고 **거꾸로 쓴 `EdCbA`** 를 한 글자씩 써 내려가야 하는 학생(디코더)이 있다. 선생님이 옆에서 연습을 시킨다.

**방법 A. 학생이 자기가 쓴 글자만 보고 이어서 쓰기 (Teacher Forcing 없음)**
1. 학생이 첫 글자를 `x`라고 잘못 쓴다. (정답은 `E`)
2. 다음 글자를 쓸 때 **자기가 쓴 틀린 `x`** 를 보고 쓴다. 이미 길을 잘못 들어서 또 틀린다.
3. 한 번 틀리면 뒤가 줄줄이 틀리고, 학생은 "어디서 뭘 잘못했는지" 배우기 어렵다. → 학습이 느리고 불안정하다.

**방법 B. 선생님이 매번 정답 글자를 보여 주기 (Teacher Forcing)**
1. 학생이 첫 글자를 `x`라고 써도, 선생님이 "정답은 `E`야"라고 고쳐 준다.
2. 다음 글자를 쓸 때는 **틀린 `x`가 아니라 정답 `E`** 를 보고 쓴다.
3. 매 칸이 "앞 글자가 정답일 때 다음 글자 맞히기"라는 **깨끗한 연습 문제**가 된다. → 학습이 빠르고 안정적이다.

이 "선생님(teacher)이 정답을 억지로 쥐여 준다(forcing)"가 **Teacher Forcing**이다.

### 구체적인 예시: 왜 "한 칸 밀어서"일까?
입력 `AbCdE`의 정답은 `EdCbA`이다. 글자를 번호로 바꾸면 (A=0, b=27, C=2, d=29, E=4, START=52):

```
정답 y            = [ E,  d,  C,  b,  A ]  = [ 4, 29,  2, 27,  0 ]
디코더 입력        = [START, E, d, C, b ]    = [52,  4, 29,  2, 27 ]
```

디코더는 각 위치에서 **그 위치의 입력을 보고 같은 위치의 정답을 맞힌다.**

| 위치(시점) | 디코더가 보는 입력 | 맞혀야 할 정답 | 의미 |
|---|---|---|---|
| 1 | START | E | 시작 신호를 받고 첫 글자를 낸다 |
| 2 | E | d | 방금 정답이었던 E를 보고 다음 글자를 낸다 |
| 3 | d | C | 〃 |
| 4 | C | b | 〃 |
| 5 | b | A | 〃 |

**왜 정답을 그대로 넣지 않고 밀어서 넣나?**
- 위치 2의 정답은 `d`이다. 입력에도 `d`를 넣어 버리면 모델은 **입력을 그대로 베끼기만 하면 만점**이다. 아무것도 배우지 않는다 (정답 유출).
- 그래서 위치 t에서는 **t보다 앞선 정답(t−1까지)만** 보여 준다. 정답을 한 칸 오른쪽으로 밀면 이 조건이 자연스럽게 만들어진다.
- 맨 앞 위치 1에는 "앞선 정답"이 없다. 비어 있으면 안 되므로 **"시작해!"라는 신호용 특수 토큰 START**를 넣는다. 실제 글자(0~51)와 겹치지 않도록 번호 52를 쓴다.

### 코드와 연결
```python
decoder_input_data = np.zeros_like(y)        # y와 같은 모양의 0 배열
decoder_input_data[:, 0] = START_TOKEN        # 모든 샘플의 첫 칸 = START(52)
decoder_input_data[:, 1:] = y[:, :-1]         # 둘째 칸부터 = 정답의 마지막 글자만 뺀 앞부분
```
- `y[:, :-1]` : 정답에서 **마지막 글자를 뺀** 것 → `[E, d, C, b]`
- `decoder_input_data[:, 1:]` : 입력에서 **첫 칸을 뺀** 나머지 자리 → 여기에 위 값을 복사
- 그래서 결과는 `[START, E, d, C, b]` 이다.
- START가 추가돼서 디코더가 받는 번호는 0~52, 즉 **53종류**다. 그래서 디코더 Embedding의 `input_dim`은 `decoder_vocab_size = 53`이고, 모델이 **맞혀야 하는 정답(출력)** 은 실제 글자 52종류뿐이라 `output_classes = 52`이다.

### 학습 vs 예측: 같은 모양이지만 입력의 출처가 다르다
| | 학습할 때 | 예측할 때 |
|---|---|---|
| 정답을 아는가 | 안다 | **모른다** |
| 디코더 입력 | **정답**을 밀어서 만든 `[START, E, d, C, b]`를 **한 번에** 통째로 | 처음엔 `[START, 0, 0, 0, 0]` |
| 진행 방식 | 5칸을 한 번에 계산 (빠름) | **한 글자씩** 예측해서 그 결과를 다음 입력에 채워 넣음 |

예측 때의 흐름 (`predict_reverse`):
1. 입력 `[START, 0, 0, 0, 0]` → 첫 글자 `E` 예측
2. 입력 `[START, E, 0, 0, 0]` → 둘째 글자 `d` 예측
3. 입력 `[START, E, d, 0, 0]` → 셋째 글자 `C` 예측 … 이렇게 `cal_len`번 반복

→ 학습 때는 선생님이 준 **정답**을 보고 쓰고, 실전(예측) 때는 **자기가 쓴 글자**를 보고 쓴다. 이 차이 때문에 학습에서는 한 번도 안 겪은 "자기 실수가 다음 입력이 되는 상황"이 실전에서 생길 수 있다. 이를 **노출 편향(exposure bias)** 이라고 부른다.

### 이전 노트북과 비교
- `5_`, `6_`, `7_` 노트북은 디코더 입력이 `np.zeros_like(X)`, 즉 **전부 0**이었다 → Teacher Forcing을 쓰지 않은 단순 구조.
- 이번 노트북은 디코더 입력에 **START + 밀어낸 정답**을 넣는다 → 진짜 Teacher Forcing. 그래서 예측 함수도 `predict` 한 번이 아니라 **글자 수만큼 반복 호출**하는 구조로 바뀌었다.
